# T4 DSA+SEA+Snapshot Cross-Dataset Runner

This notebook assumes the preprocessed Drive files already exist and only loads them for training.

Required legacy sfreq100 files:
- `/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100/cho2017.npz`
- `/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100/lee2019.npz`

Expected shapes from the existing cross-dataset matrix: Cho2017 `(10520, 64, 201)`, Lee2019 `(10800, 62, 201)`. Do not use a Lee2019 `(5400, 62, 201)` one-session export.

Runtime: select `T4 GPU` before running.

## 1. Runtime And Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
!nvidia-smi

## 2. Clone Or Update Repository

In [ ]:
import os

REPO_URL = 'https://github.com/heegyukim4043/MI_loso_project.git'
REPO_DIR = '/content/MI_loso_project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}/project/crossdata/models
!git rev-parse --short HEAD

## 3. Set Data Path And Verify Files

No MOABB download or preprocessing is run here. The notebook stops if the two `.npz` files are missing or malformed.

In [ ]:
import os
import numpy as np

os.environ['MI_PREPROCESSED_DIR'] = '/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100'
os.environ['MI_BACKUP_DIR'] = '/content/drive/MyDrive/MI_loso_project/colab_results/dsa_sea_snapshot_20260628'
os.environ['MI_N_TIMES'] = '201'

prep = os.environ['MI_PREPROCESSED_DIR']
print('MI_PREPROCESSED_DIR =', prep)
print('MI_BACKUP_DIR =', os.environ['MI_BACKUP_DIR'])
!ls -lh "$MI_PREPROCESSED_DIR"

expected = {
    'cho2017': {'shape': (10520, 64, 201), 'subjects': 52, 'trials_per_subject': None, 'sfreq': 100.0},
    'lee2019': {'shape': (10800, 62, 201), 'subjects': 54, 'trials_per_subject': 200, 'sfreq': 100.0},
}

for name, spec in expected.items():
    path = os.path.join(prep, f'{name}.npz')
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    d = np.load(path, allow_pickle=True)
    X = d['X']
    y = d['y']
    subjects = d['subjects']
    ch_names = d['ch_names']
    sfreq = float(d['sfreq'])
    unique_subjects, counts = np.unique(subjects, return_counts=True)
    labels = np.unique(y, return_counts=True)
    print(f'\n{name}: X={X.shape}, y={y.shape}, subjects={len(unique_subjects)}, channels={len(ch_names)}, sfreq={sfreq}')
    print('labels:', labels)
    print('trials/subject:', sorted(set(counts.tolist()))[:10], '...')
    if tuple(X.shape) != spec['shape']:
        raise ValueError(f'{name} has X={X.shape}, expected {spec["shape"]}. Use the legacy sfreq100 preprocessed files, not a one-session MOABB export.')
    if len(y) != X.shape[0] or len(subjects) != X.shape[0]:
        raise ValueError(f'{name} length mismatch: X={len(X)} y={len(y)} subjects={len(subjects)}')
    if abs(sfreq - spec['sfreq']) > 1e-6:
        raise ValueError(f'{name} has sfreq={sfreq}, expected {spec["sfreq"]}')
    if len(unique_subjects) != spec['subjects']:
        raise ValueError(f'{name} has {len(unique_subjects)} subjects, expected {spec["subjects"]}')
    if spec['trials_per_subject'] is not None and not np.all(counts == spec['trials_per_subject']):
        raise ValueError(f'{name} trial counts are {sorted(set(counts.tolist()))}, expected all {spec["trials_per_subject"]}')


## 4. Dependency Sanity Check

Colab already includes the core stack. This cell only checks imports.

In [ ]:
import torch, scipy, sklearn
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
print('scipy', scipy.__version__)
print('sklearn', sklearn.__version__)
!python -m py_compile manage_colab_dsa_sea_snapshot_20260628.py cross_dataset.py

## 5. Run CSPNet Cho -> Lee

Set `FORCE=True` only when rerunning after a failed or interrupted attempt.

In [ ]:


!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models cspnet \
  --direction cho2lee \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 6. Save Current Results To Drive

In [ ]:
SAVE_DIR = os.environ['MI_BACKUP_DIR']
!mkdir -p "$SAVE_DIR"
!cp -v /content/MI_loso_project/project/crossdata/colab_dsa_sea_snapshot_20260628.md "$SAVE_DIR"/ 2>/dev/null || true
!cp -v /content/MI_loso_project/project/crossdata/results/loso_results_20260628_colab_dsa_sea_snapshot_*_cross_*.csv "$SAVE_DIR"/ 2>/dev/null || true
!ls -lh "$SAVE_DIR"

## 7. Run CSPNet Lee -> Cho

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models cspnet \
  --direction lee2cho \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 8. Optional: EEGNet Both Directions

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models eegnet \
  --direction both \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 9. Optional: Conformer Both Directions

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models conformer \
  --direction both \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 10. Logs And Summary

In [ ]:
%cd /content/MI_loso_project/project/crossdata
!cat colab_dsa_sea_snapshot_20260628.md || true
!ls -lh results/runs/*dsa_sea_snapshot*.log 2>/dev/null || true
!tail -n 80 results/runs/*dsa_sea_snapshot*.log 2>/dev/null || true
!ls -lh results/loso_results_20260628_colab_dsa_sea_snapshot_*_cross_*.csv 2>/dev/null || true

## 11. Push Verified Results To GitHub

Run this only after checking the backup folder contains the expected summary and CSV files. This step is intentionally manual and asks for a GitHub token at runtime.

In [ ]:
from getpass import getpass
import os

if not os.environ.get('GITHUB_TOKEN'):
    os.environ['GITHUB_TOKEN'] = getpass('GitHub token with contents:write permission: ')

%cd /content/MI_loso_project
!git pull --ff-only origin master
!mkdir -p project/crossdata/results
!cp -v "$MI_BACKUP_DIR"/colab_dsa_sea_snapshot_20260628.md project/crossdata/colab_dsa_sea_snapshot_20260628.md
!cp -v "$MI_BACKUP_DIR"/loso_results_20260628_colab_dsa_sea_snapshot_*.csv project/crossdata/results/
!git status --short
!git config user.name "heegyukim4043"
!git config user.email "55726335+heegyukim4043@users.noreply.github.com"
!git add project/crossdata/colab_dsa_sea_snapshot_20260628.md project/crossdata/results/loso_results_20260628_colab_dsa_sea_snapshot_*.csv
!git commit -m "Add Colab DSA SEA Snapshot results" || true
!git -c http.extraHeader="AUTHORIZATION: bearer $GITHUB_TOKEN" push origin master